### Build testgen Python Package

In [1]:
!pip -q install -e ../.

### Imports

In [2]:
import pandas as pd
import numpy as np
from langchain_openai import AzureChatOpenAI
import json
import re
from tqdm.notebook import tqdm
from testgen.utils import get_examples_from_df
from pathlib import Path
from datetime import datetime as dt

### EDA

In [3]:
base_path = Path().cwd().parent

In [4]:
# # read data and rename columns
# df = pd.read_csv(base_path / "data/requirements_full.csv")
# df.columns = ["requirement", "s1", "s2", "s3", "s4", "s5", "s6"]
df = pd.read_excel(base_path / "data/requirements.xlsx")

In [5]:
# make sure all columns except the first one are binary
for c in df.columns[1:]:
    assert df[c].drop_duplicates().shape[0] == 2

# Drop Sensor

In [6]:
# # drop vihecal speed sensor
# df = df.loc[df["s6"]!=1]
# # df.drop(columns="s6", inplace=True)

# Find Examples

In [7]:
N_EXAMPLES = 1

indexes_to_drop, examples = get_examples_from_df(df, N_EXAMPLES)

In [8]:
# join all examples in a text format to add to prompt
examples_txt = ""

for e1 in examples.values():
    for e2 in e1:
        examples_txt += f"Requirement: {e2[0]}\n"
        examples_txt += f"Target Sensor/s: {e2[1]}\n"
        examples_txt += "\n"

In [9]:
print("\n".join(examples_txt.split("\n")[:8]))

Requirement: The electronic throttle control system must include redundant circuits to ensure continued operation if one circuit fails
Target Sensor/s: [ACP]

Requirement: An onboard diagnostic system must monitor for and alert to significant wear or impending failure in steering components
Target Sensor/s: [WSA]

Requirement: ABS and traction control systems must be fully integrated with the wheel speed sensors, ensuring that sensor data is used effectively for vehicle stability interventions
Target Sensor/s: [WS]


In [10]:
# drop indexes used in examples
df.drop(index=indexes_to_drop, inplace=True)

# LLM

In [11]:
from testgen.prompts import SystemPrompt
from testgen.prompts import Sensors
from testgen.prompts import UserPrompt

In [12]:
# print(SystemPrompt.format(sensors=Sensors,examples=examples_txt))

In [13]:
# llm
llm = AzureChatOpenAI(
    deployment_name="gpt-4o-mini",
    temperature=0.,
    api_version="2024-06-01"
)

In [15]:
# llm.invoke("Hi")

# Run for all Requirements

In [17]:
from testgen.utils import invoke_instance

In [23]:
results = []
responses = []

for instance in tqdm(df.iterrows()):
    result, response = invoke_instance(llm, df, SystemPrompt, Sensors, examples_txt, UserPrompt, instance)
    
    results.append(result)
    responses.append(response)

0it [00:00, ?it/s]

In [24]:
accuracy = 0
total_tokens = 0
total_completion_tokens = 0

for r in results:
    accuracy += r["accuracy"]
    total_tokens += r["total_tokens"]
    total_completion_tokens += r["completion_tokens"]

number_of_reqs = len(results)
accuracy /= len(results)
avg_token_per_req = total_tokens / len(results)
avg_completion_token_per_req = total_completion_tokens / len(results)

In [25]:
accuracy

0.6271929824561403

In [129]:
# save results
time = dt.now()

results_path = "results/conv_{model}_n-{examples}_acc-{accuracy}_{time}.json"
results_path = results_path.format(
    model=llm.deployment_name,
    examples=N_EXAMPLES,
    time=time.strftime('%m.%d.%Y-%H:%M:%S'),
    accuracy=round(accuracy, 3)
)

results_file = base_path / results_path
results_file.parent.mkdir(exist_ok=True)
results_file.touch()

with results_file.open("w") as f:
    json.dump({"accuracy": accuracy,
        "number_of_reqs": number_of_reqs,
        "total_tokens": total_tokens,
        "total_completion_tokens": total_completion_tokens,
        "avg_token_per_req": avg_token_per_req,
        "avg_completion_token_per_req": avg_completion_token_per_req,
        "responses": results}, f, indent=4)